# Python and Market Data Fundamentals

Audience: Python learners preparing for quant interviews. Prerequisites: basic Python and pandas.

Goals: Lists, dictionaries, DataFrames, time series, and validation.

Outline: setup, short tasks, checks, pitfalls. Run cells from top to bottom. All market data here is synthetic.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from market_simulator import (MarketMaker, ClientModel, MarketEnvironment,
    QuotedTrade, CompletedTrade, generate_market_data, generate_rfqs, run_simulation, make_quote)
from market_simulator.risk import beta, historical_volatility

# Resolve synthetic data from the installed local package, independent of kernel CWD.
import market_simulator
DATA = Path(market_simulator.__file__).resolve().parents[1] / 'data'
prices = pd.read_csv(DATA / 'market_prices.csv', parse_dates=['timestamp'])
requests = pd.read_csv(DATA / 'client_rfqs.csv', parse_dates=['timestamp'])
print(prices.head(2).to_string(index=False))


## Sample requests

Return the first n records as dictionaries in original order. For n <= 0, return an empty list. Example: [{'ticker':'A'}, {'ticker':'B'}], 1 -> [{'ticker':'A'}].


In [ ]:
def sample_requests(records: list[dict], n: int) -> list[dict]:
    return records[:max(0, n)]

**Reasoning and complexity:** O(min(n, len(records))) time and space for the slice. A loop also works; remember that assignment does not copy a list.

**Common mistake:** Forgetting the stated side, time ordering, or validation contract. An explicit loop is often a useful alternative when debugging.

In [ ]:
assert sample_requests([{'ticker':'A'}, {'ticker':'B'}], 1) == [{'ticker':'A'}]
assert sample_requests([], 10) == []

## Validate market data

Return True only if required columns timestamp, ticker, mid exist, every mid is finite and positive, and (timestamp,ticker) keys are unique. Empty rows are allowed if columns exist.


In [ ]:
def valid_market(frame: pd.DataFrame) -> bool:
    required = {'timestamp', 'ticker', 'mid'}
    return (required.issubset(frame.columns) and frame['mid'].notna().all()
            and np.isfinite(frame['mid']).all() and (frame['mid'] > 0).all()
            and frame['ticker'].notna().all() and frame['timestamp'].notna().all()
            and not frame.duplicated(['timestamp', 'ticker']).any())

**Reasoning and complexity:** O(n) time and O(n) temporary space in pandas. Check columns before indexing; duplicate keys make joins ambiguous.

**Common mistake:** Forgetting the stated side, time ordering, or validation contract. An explicit loop is often a useful alternative when debugging.

In [ ]:
assert valid_market(prices)
assert not valid_market(pd.DataFrame({'timestamp':[1,1], 'ticker':['A','A'], 'mid':[1,2]}))

## Group and return

For one ticker, sort by timestamp and return a Series of simple returns (P_t/P_(t-1)-1), excluding the first missing value. Example prices 100, 110 -> [0.10].


In [ ]:
def ticker_returns(frame: pd.DataFrame, ticker: str) -> pd.Series:
    series = frame.loc[frame['ticker'] == ticker].sort_values('timestamp')['mid']
    return series.pct_change().dropna()

**Reasoning and complexity:** O(n log n) from sorting; filtering is O(n). Avoid computing returns before sorting by time.

**Common mistake:** Forgetting the stated side, time ordering, or validation contract. An explicit loop is often a useful alternative when debugging.

In [ ]:
assert np.isclose(ticker_returns(pd.DataFrame({'ticker':['A','A'], 'timestamp':[1,2], 'mid':[100,110]}), 'A').iloc[0], .1)
assert ticker_returns(prices, 'MISSING').empty

## Extension
Try a different random seed or add a transaction fee. Explain how your answer changes.